In [5]:
import pandas as pd

# data_middle.csv 파일 불러옴
data_middle = pd.read_csv(
    r"..\..\data\processed\data_middle.csv",
    index_col=0,
    parse_dates=True
)

# Signal1: MA5와 MA15의 골든크로스와 데드크로스
data_middle['Signal1'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_MA5'].shift(1) <= data_middle['KOSPI 200_MA15'].shift(1)) & (data_middle['KOSPI 200_MA5'] > data_middle['KOSPI 200_MA15'])
sell_cond = (data_middle['KOSPI 200_MA5'].shift(1) >= data_middle['KOSPI 200_MA15'].shift(1)) & (data_middle['KOSPI 200_MA5'] < data_middle['KOSPI 200_MA15'])

data_middle.loc[buy_cond, 'Signal1'] = 'Buy'
data_middle.loc[sell_cond, 'Signal1'] = 'Sell'

# Signal2: EMA5와 EMA15의 골든크로스와 데드크로스
data_middle['Signal2'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_EMA5'].shift(1) <= data_middle['KOSPI 200_EMA15'].shift(1)) & (data_middle['KOSPI 200_EMA5'] > data_middle['KOSPI 200_EMA15'])
sell_cond = (data_middle['KOSPI 200_EMA5'].shift(1) >= data_middle['KOSPI 200_EMA15'].shift(1)) & (data_middle['KOSPI 200_EMA5'] < data_middle['KOSPI 200_EMA15'])

data_middle.loc[buy_cond, 'Signal2'] = 'Buy'
data_middle.loc[sell_cond, 'Signal2'] = 'Sell'

# Signal3: RSI14의 과매도와 과매수 신호에 따른 매매 신호
data_middle['Signal3'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_RSI14'].shift(1) <= 30) & (data_middle['KOSPI 200_RSI14'] > 30)
sell_cond = (data_middle['KOSPI 200_RSI14'].shift(1) >= 70) & (data_middle['KOSPI 200_RSI14'] < 70)

data_middle.loc[buy_cond, 'Signal3'] = 'Buy'
data_middle.loc[sell_cond, 'Signal3'] = 'Sell'

# Signal4: Bollinger Bands 기반의 매매 신호
data_middle['Signal4'] = 'Stay'

buy_cond = (data_middle['KOSPI 200 Close'].shift(1) <= data_middle['KOSPI 200_BB_LOWER15'].shift(1)) & (data_middle['KOSPI 200 Close'] > data_middle['KOSPI 200_BB_LOWER15'])
sell_cond = (data_middle['KOSPI 200 Close'].shift(1) >= data_middle['KOSPI 200_BB_UPPER15'].shift(1)) & (data_middle['KOSPI 200 Close'] < data_middle['KOSPI 200_BB_UPPER15'])

data_middle.loc[buy_cond, 'Signal4'] = 'Buy'
data_middle.loc[sell_cond, 'Signal4'] = 'Sell'

In [9]:
# 각 신호 분포를 가로 테이블로 보기
signal_cols = ['Signal1', 'Signal2', 'Signal3', 'Signal4']

table = (
    pd.DataFrame({
        col: data_middle[col].value_counts(dropna=False)
        for col in signal_cols
    })
    .fillna(0)
    .astype(int)
    .reindex(['Stay', 'Buy', 'Sell'])   # 행 순서 고정
)

display(table)

,Signal1,Signal2,Signal3,Signal4
Stay,3797,3811,3855,3869
Buy,156,149,98,131
Sell,156,149,156,109


In [10]:
signal_cols = ['Signal1', 'Signal2', 'Signal3', 'Signal4']

# 1. 각 Signal의 범주 순서를 지정
for col in signal_cols:
    data_middle[col] = pd.Categorical(
        data_middle[col],
        categories=['Stay', 'Buy', 'Sell']
    )

# 2. Stay를 기준범주로 하여 가변수 생성
signal_dummies = pd.get_dummies(
    data_middle[signal_cols],
    prefix=signal_cols,
    prefix_sep='_',
    drop_first=True,
    dtype=int
)

# 3. 원본 데이터프레임에 붙이기
data_middle = pd.concat([data_middle, signal_dummies], axis=1)

data_middle = data_middle.drop(columns=['Signal1', 'Signal2', 'Signal3', 'Signal4'])

cols = [col for col in data_middle.columns if col != 'Risk_Label'] + ['Risk_Label']
data_middle = data_middle[cols]

print(data_middle.head())
print(list(data_middle.columns))

            Shanghai Comp  KODEX 200  TOPIX  Brent Crude Oil   Gold Spot  \
Date                                                                       
2009-04-17    2503.935059    17370.0  875.0        51.959999  867.400024   
2009-04-20    2557.456055    17480.0  876.0        51.959999  887.000000   
2009-04-21    2535.827881    17480.0  855.0        51.959999  882.099976   
2009-04-22    2461.345947    17715.0  856.0        51.959999  891.799988   
2009-04-23    2463.954102    17895.0  862.0        51.959999  905.900024   

            JPY/KRW      USD/KRW       NASDAQ      KOSDAQ  KOSPI 200 Close  \
Date                                                                         
2009-04-17   13.371  1325.800049  1673.069946  483.799988       171.330002   
2009-04-20   13.536  1327.500000  1608.209961  491.940002       172.300003   
2009-04-21   13.727  1354.300049  1643.849976  497.190002       171.960007   
2009-04-22   13.726  1346.599976  1646.119995  509.899994       174.399994   